# Imports and Configs

In [1]:
import os
from src.config import settings

import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from src.data.loader import RetailRocketLoader
from src.data.preprocessor import EventWeightPreprocessor, MinInteractionsFilter
from src.features.engineering import encode_ids, split_interactions
from src.training.trainer import run_training

c:\Users\giova\Documents\FIAP - MLE\2 - Big Data Architecture\retailrocket-recsys\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
settings.mlflow_tracking_uri = "file:./mlruns"

In [3]:
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\giova\Documents\FIAP - MLE\2 - Big Data Architecture\retailrocket-recsys


In [4]:
# try:
#     import torch_directml
#     # Tenta obter o dispositivo de aceleração do DirectML
#     device = torch_directml.device()
# except ImportError:
#     # Fallback caso a biblioteca não esteja instalada no ambiente
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Executando no dispositivo: {device}")

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {device}")

Executando no dispositivo: cpu


# Load Data

In [6]:
loader = RetailRocketLoader()
events_df = loader.load_events()
items_df = loader.load_item_properties()

INFO: RetailRocketLoader initialized with path: data\raw
INFO: Loading events from: data\raw\events.csv
INFO: Raw events loaded: 2756101 rows
INFO: Events validated: 2756101 → 2756101 rows | distribution: {'view': 2664312, 'addtocart': 69332, 'transaction': 22457}
INFO: Loading item properties from 2 files
INFO: Item properties loaded: 20275902 rows, 417053 unique items


In [7]:
events_df

,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,NaN
1,2015-06-02 05:50:14.164,992329,view,248676,NaN
2,2015-06-02 05:13:19.827,111016,view,318965,NaN
3,2015-06-02 05:12:35.914,483717,view,253185,NaN
4,2015-06-02 05:02:17.106,951259,view,367447,NaN
...,...,...,...,...,...
2756096,2015-08-01 03:13:05.939,591435,view,261427,NaN
2756097,2015-08-01 03:30:13.142,762376,view,115946,NaN
2756098,2015-08-01 02:57:00.527,1251746,view,78144,NaN
2756099,2015-08-01 03:08:50.703,1184451,view,283392,NaN


In [8]:
events_df.event.unique()

array(['view', 'addtocart', 'transaction'], dtype=object)

In [9]:
items_df

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513
...,...,...,...,...
20275897,1433646000000,236931,929,n12.000
20275898,1440903600000,455746,6,150169 639134
20275899,1439694000000,347565,686,610834
20275900,1433646000000,287231,867,769062


# Preprocessing

### Weights

In [10]:
# 1. Instanciar as classes de pré-processamento
weight_preprocessor = EventWeightPreprocessor()
interactions_filter = MinInteractionsFilter(min_user=5, min_item=5)

# 2. Aplicar o mapeamento de pesos e agrupamento (soma dos scores por user-item)
print("Aplicando mapeamento de pesos...")
events_weighted = weight_preprocessor.transform(events_df)

# 3. Aplicar a filtragem de interações mínimas (5)
print("\nFiltrando interações mínimas...")
events_filtered = interactions_filter.transform(events_weighted)

# Visualizar as primeiras linhas do resultado filtrado
events_filtered.head()


INFO: MinInteractionsFilter initialized: min_user=5, min_item=5
INFO: EventWeightPreprocessor: applying weights {'view': 1, 'addtocart': 3, 'transaction': 5} to 2756101 events


Aplicando mapeamento de pesos...


INFO: EventWeightPreprocessor: aggregated to 2145179 (user, item) pairs | score range: [1.0, 308.0]



Filtrando interações mínimas...


INFO: MinInteractionsFilter: 2145179 → 312680 rows | users: 1407580 → 38178 | items: 235061 → 23188


,visitorid,itemid,score
0,51,49967,1
1,51,198762,2
2,51,429304,1
3,54,38965,2
4,54,249114,1


### IDs Encoding

In [11]:
# 1. Executar o mapeamento de IDs (Categorificação)
encoded_data = encode_ids(events_filtered)

print("Mapeamento concluído:")
print(f"  Número de usuários únicos: {encoded_data.num_users}")
print(f"  Número de itens únicos: {encoded_data.num_items}")

# 2. Dividir os dados em treino, validação e teste
splits = split_interactions(encoded_data.df, val_size=0.1, test_size=0.1)

print("\nDivisão dos dados concluída:")
print(f"  Treino:    {len(splits.train)} interações")
print(f"  Validação: {len(splits.val)} interações")
print(f"  Teste:     {len(splits.test)} interações")

# Inspecionar o formato dos dados mapeados no treino
splits.train.head()

Mapeamento concluído:
  Número de usuários únicos: 38178
  Número de itens únicos: 23188

Divisão dos dados concluída:
  Treino:    250143 interações
  Validação: 31269 interações
  Teste:     31268 interações


,visitorid,itemid,score,user_idx,item_idx
0,625866,361103,1,17043,17949
1,1015139,329615,1,27557,16355
2,18483,546,1,501,31
3,404403,180766,1,10917,8930
4,542702,369447,1,14688,18367


### Negative Sampling

In [12]:


def generate_static_negatives(df, num_items, n_negatives=4, seed=42): #4 negatives for each positive is an industry standard
    """
    Gera amostras negativas (score = 0.0) para cada usuário no DataFrame.
    Garante que os itens amostrados não tenham interações anteriores com o respectivo usuário.
    """
    np.random.seed(seed)
    
    # 1. Mapeia os itens positivos (com os quais o usuário já interagiu)
    user_positives = df.groupby('user_idx')['item_idx'].apply(set).to_dict()
    
    neg_users = []
    neg_items = []
    neg_scores = []
    
    # 2. Gera os negativos para cada usuário ativo no conjunto
    for user, pos_items in user_positives.items():
        n_to_sample = n_negatives * len(pos_items) #4 positives for each positive interaction the user had
        samples = []
        
        # Loop para garantir que sorteamos a quantidade correta de itens válidos (não-interagidos)
        while len(samples) < n_to_sample:
            # Sorteia candidatos aleatórios
            candidates = np.random.randint(0, num_items, size=n_to_sample - len(samples))
            for c in candidates:
                if c not in pos_items:
                    samples.append(c)
        
        neg_users.extend([user] * n_to_sample)
        neg_items.extend(samples)
        neg_scores.extend([0.0] * n_to_sample)
        
    # 3. Cria o DataFrame dos negativos
    neg_df = pd.DataFrame({
        'user_idx': neg_users,
        'item_idx': neg_items,
        'score': neg_scores
    })
    
    # 4. Combina positivos e negativos e embaralha os dados
    pos_df = df[['user_idx', 'item_idx', 'score']].copy()
    final_df = pd.concat([pos_df, neg_df], ignore_index=True)
    final_df = final_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    
    return final_df

# Executa a amostragem de negativos para a base de TREINO
num_items = encoded_data.num_items
train_df_with_negs = generate_static_negatives(splits.train, num_items=num_items, n_negatives=4)

print(f"Base de Treino Original (Positivos): {len(splits.train)} linhas")
print(f"Base de Treino com Negativos:        {len(train_df_with_negs)} linhas")
print("\nDistribuição dos targets:")
print(train_df_with_negs['score'].value_counts())


Base de Treino Original (Positivos): 250143 linhas
Base de Treino com Negativos:        1250715 linhas

Distribuição dos targets:
score
0.0      1000572
1.0       181489
2.0        31293
3.0        12015
4.0         7249
          ...   
55.0           1
73.0           1
161.0          1
66.0           1
170.0          1
Name: count, Length: 77, dtype: int64


### Dataloaders

In [13]:
class InteractionDataset(Dataset):
    """
    Dataset PyTorch customizado. Converte as colunas do Pandas em Tensores do PyTorch
    e permite acesso indexado aos pares usuário-item e seus respectivos scores.
    """
    def __init__(self, df: pd.DataFrame):
        # Usamos torch.long (int64) para os IDs porque eles serão usados como índices nas camadas de Embedding - vão se tornar um índice de lookup
        self.user_ids = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_idx'].values, dtype=torch.long)
        # Usamos torch.float32 para o score porque ele será o target contínuo da perda (MSE)
        self.scores = torch.tensor(df['score'].values, dtype=torch.float32)
        
    def __len__(self):
        # Retorna o número total de linhas/exemplos
        return len(self.scores)
        
    def __getitem__(self, idx):
        # Retorna a tupla (usuario, item, score) correspondente ao índice
        return self.user_ids[idx], self.item_ids[idx], self.scores[idx]


# 1. Instanciar os Datasets PyTorch
train_dataset = InteractionDataset(train_df_with_negs)
val_dataset = InteractionDataset(splits.val)
test_dataset = InteractionDataset(splits.test)

# 2. Criar os DataLoaders
# O Batch Size padrão de 512 é excelente para recomendação (processa rápido)
BATCH_SIZE = 8192

# No treino, shuffle=True embaralha os dados no início de cada época
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# Na validação/teste, shuffle=False (só queremos avaliar em ordem estável)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Testar o DataLoader pegando o primeiro lote (batch) de treino
first_batch = next(iter(train_loader))
users, items, scores = first_batch

print("Inspecionando o primeiro Batch do DataLoader:")
print(f"  Shape dos Usuários: {users.shape} | Tipo: {users.dtype}")
print(f"  Shape dos Itens:    {items.shape} | Tipo: {items.dtype}")
print(f"  Shape dos Scores:   {scores.shape} | Tipo: {scores.dtype}")


Inspecionando o primeiro Batch do DataLoader:
  Shape dos Usuários: torch.Size([8192]) | Tipo: torch.int64
  Shape dos Itens:    torch.Size([8192]) | Tipo: torch.int64
  Shape dos Scores:   torch.Size([8192]) | Tipo: torch.float32


# Neural Network

In [14]:
class MatrixFactorization(nn.Module):
    """
    Modelo de Fatoração de Matrizes (Collaborative Filtering) com Biases.
    Previsão = (User_emb · Item_emb) + User_bias + Item_bias + Global_bias
    """
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        
        # 1. Tabelas de Embeddings para Usuários e Itens
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        
        # 2. Tabelas de Vieses (Biases) para Usuários e Itens
        # Cada viés é um único número por ID (por isso a dimensão é 1)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        
        # 3. Viés Global (um único parâmetro numérico treinável)
        self.global_bias = nn.Parameter(torch.zeros(1))
        
        # Inicialização dos pesos
        self._init_weights()
        
    def _init_weights(self):
        # Inicializa embeddings com valores aleatórios pequenos (ajuda na convergência)
        # Modelo começa prevendo a média geral dos dados, ao invés de inicializar todos os pesos como 0
        nn.init.normal_(self.user_embeddings.weight, std=0.01)
        nn.init.normal_(self.item_embeddings.weight, std=0.01)
        
        # Inicializa os vieses locais com zero
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        
    def forward(self, user_ids, item_ids):
        # 1. Recupera os embeddings correspondentes aos IDs do lote
        user_emb = self.user_embeddings(user_ids)  # Formato: [batch_size, embedding_dim]
        item_emb = self.item_embeddings(item_ids)  # Formato: [batch_size, embedding_dim]
        
        # 2. Recupera os vieses e remove a dimensão extra de tamanho 1 (squeeze)
        # Ex: converte formato [batch_size, 1] para [batch_size]
        user_b = self.user_bias(user_ids).squeeze()
        item_b = self.item_bias(item_ids).squeeze()
        
        # 3. Calcula o produto escalar (dot product) para cada par do batch
        # Multiplica os embeddings elemento por elemento e depois soma na dimensão das características (dim=1)
        dot_product = (user_emb * item_emb).sum(dim=1)  # Formato: [batch_size]
        
        # 4. Soma tudo: Produto Escalar + Viés Usuário + Viés Item + Viés Global
        prediction = dot_product + user_b + item_b + self.global_bias
        
        return prediction


In [15]:
model = MatrixFactorization(
    num_users=encoded_data.num_users, 
    num_items=encoded_data.num_items, 
    embedding_dim=64
)

In [16]:
trained_model = run_training(
    model, 
    train_loader, 
    val_loader, 
    run_name="matrix_factorization", 
    device=device
)

INFO: Training on device: cpu
INFO: Epoch 001 | train_loss: 2.1059 | val_loss: 9.7123
INFO: Epoch 002 | train_loss: 2.0046 | val_loss: 9.2332
INFO: Epoch 003 | train_loss: 1.9107 | val_loss: 8.8454
INFO: Epoch 004 | train_loss: 1.7963 | val_loss: 8.5473
INFO: Epoch 005 | train_loss: 1.6784 | val_loss: 8.3369
INFO: Epoch 006 | train_loss: 1.5629 | val_loss: 8.1819
INFO: Epoch 007 | train_loss: 1.4516 | val_loss: 8.0637
INFO: Epoch 008 | train_loss: 1.3475 | val_loss: 7.9715
INFO: Epoch 009 | train_loss: 1.2521 | val_loss: 7.9035
INFO: Epoch 010 | train_loss: 1.1639 | val_loss: 7.8514
INFO: Epoch 011 | train_loss: 1.0828 | val_loss: 7.8139
INFO: Epoch 012 | train_loss: 1.0091 | val_loss: 7.7880
INFO: Epoch 013 | train_loss: 0.9417 | val_loss: 7.7710
INFO: Epoch 014 | train_loss: 0.8887 | val_loss: 7.7565
INFO: Epoch 015 | train_loss: 0.8231 | val_loss: 7.7515
INFO: Epoch 016 | train_loss: 0.7709 | val_loss: 7.7489
INFO: Epoch 017 | train_loss: 0.7232 | val_loss: 7.7485
INFO: Epoch 018 | 

In [17]:
import os
import numpy as np
import torch
from src.evaluation.metrics import compute_all_metrics

# ==========================================
# PASSO 1: Salvar backup do modelo em arquivo físico
# ==========================================
os.makedirs("models", exist_ok=True)
checkpoint_path = "models/matrix_factorization.pth"
torch.save(trained_model.state_dict(), checkpoint_path)
print(f"-> Sucesso: O modelo físico foi salvo em: {checkpoint_path}")
print("Você pode desligar o computador sem se preocupar. Para recarregar depois, basta rodar:")
print("model.load_state_dict(torch.load('models/matrix_factorization.pth'))\n")


# ==========================================
# PASSO 2: Avaliação de Performance (Ranking Top-10)
# ==========================================
print("Iniciando avaliação de ranking em 200 usuários de teste...")
trained_model.eval()

k = 10
_EVAL_USERS = 200  # Limite para a avaliação rodar rápido

# 1. Identificar itens vistos no treino para não recomendá-los novamente (evitar redundância)
seen_in_train = splits.train.groupby("user_idx")["item_idx"].apply(set).to_dict()

# 2. Obter os itens reais que os usuários interagiram no conjunto de teste (gabarito)
ground_truth = splits.test.groupby("user_idx")["item_idx"].apply(set).to_dict()

# 3. Amostrar os usuários de teste
test_users = list(ground_truth.keys())[:_EVAL_USERS]

# Prepara a lista de todos os itens disponíveis no sistema
all_items = torch.arange(encoded_data.num_items, dtype=torch.long, device=device)
recommendations = {}

with torch.no_grad():
    for uid in test_users:
        # Repete o ID do usuário para bater com o tamanho da lista de itens
        user_tensor = torch.full((encoded_data.num_items,), uid, dtype=torch.long, device=device)
        
        # Faz as previsões para todos os itens do catálogo
        scores = trained_model(user_tensor, all_items).squeeze().cpu().numpy()
        
        # Filtra os itens que o usuário já viu no treino (mascara com score negativo infinito)
        user_seen = seen_in_train.get(uid, set())
        for item in user_seen:
            if item < encoded_data.num_items:
                scores[item] = -np.inf
                
        # Pega o Top-10 itens com maiores scores
        top_k = np.argsort(scores)[::-1][:k].tolist()
        recommendations[uid] = top_k

# 4. Calcular e exibir as métricas de ranking do modelo
metrics_df = compute_all_metrics(recommendations, ground_truth, k=k)
print("\n=== Resultados da Avaliação (Top-10) ===")
metrics_df


-> Sucesso: O modelo físico foi salvo em: models/matrix_factorization.pth
Você pode desligar o computador sem se preocupar. Para recarregar depois, basta rodar:
model.load_state_dict(torch.load('models/matrix_factorization.pth'))

Iniciando avaliação de ranking em 200 usuários de teste...

=== Resultados da Avaliação (Top-10) ===


,mean_score
precision@10,0.002500
recall@10,0.018750
ndcg@10,0.010361
hit_rate@10,0.025000
